# Analyze HATR Result Folders

This Colab notebook compares HATR experiments under `/content/drive/MyDrive/DCASE2026`.

It is designed for folders such as:

- `hatr_bsd10k_colab`
- `hatr_bsd10k_colab_no_conf1`
- `hatr_bsd10k_colab_select_class_confi1`

It compares fold-level metrics, mean ± std metrics, class distributions, and per-class prediction behavior when prediction CSVs are available.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

ROOT = Path('/content/drive/MyDrive/DCASE2026')
REPORT_DIR = ROOT / 'hatr_result_analysis_report'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATE_EXPERIMENTS = {
    'all_confidence': ROOT / 'hatr_bsd10k_colab',
    'no_confidence1': ROOT / 'hatr_bsd10k_colab_no_conf1',
    'select_confidence1_classes': ROOT / 'hatr_bsd10k_colab_select_class_confi1',
}

EXPERIMENTS = {name: path for name, path in CANDIDATE_EXPERIMENTS.items() if path.exists()}
print('Found experiments:')
for name, path in EXPERIMENTS.items():
    print(f'  {name}: {path}')

if not EXPERIMENTS:
    raise FileNotFoundError('No expected HATR result folders found under /content/drive/MyDrive/DCASE2026')

In [ ]:
# Locate useful files in each experiment
def find_first(root, names):
    for name in names:
        hits = list(root.rglob(name))
        if hits:
            return hits[0]
    return None

def experiment_layout(path):
    return {
        'fold_summary': find_first(path, ['fold_summary.csv', 'summary_audio_test.csv', 'summary.csv']),
        'mean_std': find_first(path, ['baseline_style_metrics_mean_std.csv', 'summary_audio_test_mean_std.csv']),
        'processed_dataset': find_first(path, ['processed_dataset.csv']),
        'class_dict': find_first(path, ['class_dict.json']),
        'top_class_dict': find_first(path, ['top_class_dict.json']),
        'predictions': sorted(path.rglob('predictions*.csv')),
        'confusion_npy': sorted(path.rglob('confusion_matrix*.npy')),
    }

layouts = {name: experiment_layout(path) for name, path in EXPERIMENTS.items()}
for name, layout in layouts.items():
    print('\n==', name, '==')
    for key, value in layout.items():
        if isinstance(value, list):
            print(key, len(value))
            for item in value[:5]:
                print(' ', item)
        else:
            print(key, value)

In [ ]:
# Fold metric comparison
METRIC_COLS = [
    'accuracy',
    'top_accuracy',
    'macro_accuracy',
    'macro_top_accuracy',
    'hierarchical_accuracy',
    'hierarchical_precision',
    'hierarchical_recall',
    'hierarchical_f1',
]

summary_frames = []
for exp_name, layout in layouts.items():
    path = layout['fold_summary']
    if path is None:
        print('missing fold summary:', exp_name)
        continue
    df = pd.read_csv(path)
    df['experiment'] = exp_name
    df['source_file'] = str(path)
    summary_frames.append(df)

all_folds = pd.concat(summary_frames, ignore_index=True) if summary_frames else pd.DataFrame()
all_folds.to_csv(REPORT_DIR / 'all_fold_metrics.csv', index=False)
display(all_folds)

available_metrics = [m for m in METRIC_COLS if m in all_folds.columns]
metric_summary = []
for exp_name, group in all_folds.groupby('experiment', sort=False):
    row = {'experiment': exp_name, 'fold_count': len(group)}
    for metric in available_metrics:
        values = pd.to_numeric(group[metric], errors='coerce').dropna()
        row[f'{metric}_mean'] = values.mean() if len(values) else np.nan
        row[f'{metric}_std'] = values.std(ddof=0) if len(values) else np.nan
        row[f'{metric}_mean_pm_std'] = f"{row[f'{metric}_mean']:.2f} ± {row[f'{metric}_std']:.2f}" if len(values) else ''
    metric_summary.append(row)

metric_summary = pd.DataFrame(metric_summary)
metric_summary.to_csv(REPORT_DIR / 'experiment_metric_mean_std.csv', index=False)
display(metric_summary)

In [ ]:
# Paired fold differences versus all_confidence
if 'all_confidence' in all_folds['experiment'].unique():
    baseline = all_folds[all_folds['experiment'] == 'all_confidence'].copy()
    diff_rows = []
    for exp_name in all_folds['experiment'].unique():
        if exp_name == 'all_confidence':
            continue
        comp = all_folds[all_folds['experiment'] == exp_name].copy()
        if 'fold' not in baseline.columns or 'fold' not in comp.columns:
            continue
        merged = baseline.merge(comp, on='fold', suffixes=('_all', f'_{exp_name}'))
        for metric in available_metrics:
            a = pd.to_numeric(merged[f'{metric}_all'], errors='coerce')
            b = pd.to_numeric(merged[f'{metric}_{exp_name}'], errors='coerce')
            delta = b - a
            diff_rows.append({
                'comparison': f'{exp_name} - all_confidence',
                'metric': metric,
                'mean_delta': delta.mean(),
                'std_delta': delta.std(ddof=0),
                'improved_folds': int((delta > 0).sum()),
                'worse_folds': int((delta < 0).sum()),
                'same_folds': int((delta == 0).sum()),
                'fold_deltas': ', '.join([f'{x:.2f}' for x in delta.tolist()]),
            })
    diff_df = pd.DataFrame(diff_rows)
    diff_df.to_csv(REPORT_DIR / 'paired_fold_differences_vs_all_confidence.csv', index=False)
    display(diff_df)
else:
    print('No all_confidence experiment found, skipping paired differences.')

In [ ]:
# Plot main metrics
plot_metrics = [m for m in ['accuracy', 'hierarchical_accuracy', 'hierarchical_f1', 'macro_accuracy'] if f'{m}_mean' in metric_summary.columns]
if plot_metrics:
    fig, axes = plt.subplots(1, len(plot_metrics), figsize=(5 * len(plot_metrics), 4), squeeze=False)
    for ax, metric in zip(axes[0], plot_metrics):
        x = np.arange(len(metric_summary))
        means = metric_summary[f'{metric}_mean'].to_numpy(dtype=float)
        stds = metric_summary[f'{metric}_std'].to_numpy(dtype=float)
        ax.bar(x, means, yerr=stds, capsize=4, color=['#4C78A8', '#F58518', '#54A24B', '#B279A2'][:len(x)])
        ax.set_xticks(x)
        ax.set_xticklabels(metric_summary['experiment'], rotation=35, ha='right')
        ax.set_title(metric)
        ax.set_ylabel('%')
        ax.grid(axis='y', alpha=0.25)
    fig.tight_layout()
    plot_path = REPORT_DIR / 'main_metric_comparison.png'
    fig.savefig(plot_path, dpi=180, bbox_inches='tight')
    plt.show()
    print('saved:', plot_path)

In [ ]:
# Dataset composition comparison, if processed_dataset.csv exists
composition_rows = []
class_count_frames = []
for exp_name, layout in layouts.items():
    path = layout['processed_dataset']
    if path is None:
        print('missing processed_dataset:', exp_name)
        continue
    df = pd.read_csv(path)
    composition_rows.append({
        'experiment': exp_name,
        'samples': len(df),
        'classes': df['class'].nunique() if 'class' in df.columns else np.nan,
        'top_classes': df['top_class'].nunique() if 'top_class' in df.columns else np.nan,
    })
    if 'class' in df.columns:
        counts = df['class'].value_counts().rename(exp_name)
        class_count_frames.append(counts)

composition = pd.DataFrame(composition_rows)
composition.to_csv(REPORT_DIR / 'dataset_composition.csv', index=False)
display(composition)

if class_count_frames:
    class_counts = pd.concat(class_count_frames, axis=1).fillna(0).astype(int)
    class_counts['max_minus_min'] = class_counts.max(axis=1) - class_counts.min(axis=1)
    class_counts = class_counts.sort_values('max_minus_min', ascending=False)
    class_counts.to_csv(REPORT_DIR / 'class_count_changes.csv')
    display(class_counts.head(30))

In [ ]:
# Per-class metrics from fold prediction CSVs, if available
def read_prediction_file(path):
    df = pd.read_csv(path)
    if {'y_true', 'y_pred'}.issubset(df.columns):
        return df[['y_true', 'y_pred']].rename(columns={'y_true': 'true_id', 'y_pred': 'pred_id'})
    if {'ground_truth', 'prediction'}.issubset(df.columns):
        return df[['ground_truth', 'prediction']].rename(columns={'ground_truth': 'true_label', 'prediction': 'pred_label'})
    return None

def load_id_to_class(layout):
    if layout['class_dict'] is None:
        return None
    with open(layout['class_dict']) as f:
        class_dict = json.load(f)
    return {int(v): k for k, v in class_dict.items()}

per_class_frames = []
for exp_name, layout in layouts.items():
    id_to_class = load_id_to_class(layout)
    pred_files = [p for p in layout['predictions'] if 'prediction' in p.name.lower()]
    if not pred_files:
        print('missing predictions:', exp_name)
        continue
    rows = []
    for path in pred_files:
        pred = read_prediction_file(path)
        if pred is None:
            continue
        if {'true_id', 'pred_id'}.issubset(pred.columns):
            pred['true_label'] = pred['true_id'].astype(int).map(id_to_class) if id_to_class else pred['true_id'].astype(str)
            pred['pred_label'] = pred['pred_id'].astype(int).map(id_to_class) if id_to_class else pred['pred_id'].astype(str)
        labels = sorted(pred['true_label'].dropna().unique())
        precision, recall, f1, support = precision_recall_fscore_support(
            pred['true_label'], pred['pred_label'], labels=labels, zero_division=0
        )
        fold_name = path.parent.name
        for label, p, r, f, s in zip(labels, precision, recall, f1, support):
            rows.append({'experiment': exp_name, 'fold': fold_name, 'class': label, 'precision': p, 'recall': r, 'f1': f, 'support': s})
    if rows:
        per_class_frames.append(pd.DataFrame(rows))

per_class = pd.concat(per_class_frames, ignore_index=True) if per_class_frames else pd.DataFrame()
per_class.to_csv(REPORT_DIR / 'per_class_fold_metrics.csv', index=False)
display(per_class.head())

if not per_class.empty:
    per_class_summary = per_class.groupby(['experiment', 'class']).agg(
        precision_mean=('precision', 'mean'),
        recall_mean=('recall', 'mean'),
        f1_mean=('f1', 'mean'),
        support_mean=('support', 'mean'),
    ).reset_index()
    per_class_summary.to_csv(REPORT_DIR / 'per_class_metric_summary.csv', index=False)
    display(per_class_summary.head())

In [ ]:
# Per-class deltas versus all_confidence
if not per_class.empty and 'all_confidence' in per_class['experiment'].unique():
    pcs = per_class.groupby(['experiment', 'class']).agg(
        recall=('recall', 'mean'),
        f1=('f1', 'mean'),
        support=('support', 'mean'),
    ).reset_index()
    base = pcs[pcs['experiment'] == 'all_confidence'].drop(columns='experiment')
    delta_frames = []
    for exp_name in pcs['experiment'].unique():
        if exp_name == 'all_confidence':
            continue
        comp = pcs[pcs['experiment'] == exp_name].drop(columns='experiment')
        merged = base.merge(comp, on='class', suffixes=('_all', f'_{exp_name}'))
        merged['experiment'] = exp_name
        merged['delta_recall'] = merged[f'recall_{exp_name}'] - merged['recall_all']
        merged['delta_f1'] = merged[f'f1_{exp_name}'] - merged['f1_all']
        delta_frames.append(merged)
    class_deltas = pd.concat(delta_frames, ignore_index=True) if delta_frames else pd.DataFrame()
    class_deltas.to_csv(REPORT_DIR / 'per_class_deltas_vs_all_confidence.csv', index=False)
    display(class_deltas.sort_values('delta_f1').head(20))
    display(class_deltas.sort_values('delta_f1', ascending=False).head(20))
else:
    print('Per-class delta skipped. Need predictions and all_confidence experiment.')

In [ ]:
# Text conclusion helper
print('Report saved to:', REPORT_DIR)
print('\nKey files:')
for name in [
    'experiment_metric_mean_std.csv',
    'paired_fold_differences_vs_all_confidence.csv',
    'dataset_composition.csv',
    'class_count_changes.csv',
    'per_class_deltas_vs_all_confidence.csv',
    'main_metric_comparison.png',
]:
    path = REPORT_DIR / name
    print(path, 'exists=' + str(path.exists()))

if 'metric_summary' in globals() and not metric_summary.empty:
    print('\nMetric summary:')
    for _, row in metric_summary.iterrows():
        print('\n', row['experiment'])
        for metric in ['accuracy', 'hierarchical_accuracy', 'hierarchical_f1']:
            col = f'{metric}_mean_pm_std'
            if col in row:
                print(f'  {metric}: {row[col]}')